In [1]:
import os
import datetime

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import h5py

# Load Data

In [3]:
date = datetime.datetime(2018,12,18)

ham_dir   = 'ham_data'
ham_fname = 'rsd{!s}.01.hdf5'.format(date.strftime('%Y-%m-%d'))
ham_fpath = os.path.join(ham_dir,ham_fname)
print(ham_fpath)

ham_data/rsd2018-12-18.01.hdf5


In [4]:
with h5py.File(ham_fpath,'r') as h5_ham:
    data     = h5_ham['Data']
    metadata = h5_ham['Metadata']
    df       = pd.DataFrame(data['Table Layout'][()])

# Rename min and sec columns to be compatible with datetime keywords.
df = df.rename(columns={'min':'minute','sec':'second'})

In [5]:
## DECODE BYTE OBJECTS TO STRINGS
# Select columns that are byte objects
str_df = df.select_dtypes([object])

# Convert all columns to strings
str_df = str_df.stack().str.decode('utf-8').unstack()

# Replace columns in original df
for col in str_df:
    df[col] = str_df[col]

# Delete str_df from memory
del str_df

In [6]:
# Convert Datetimes
df['date'] = pd.to_datetime(df[['year','month','day','hour','minute','second']])

In [7]:
# Convert Timestamps
df['ut1_unix_LOCAL'] = df['ut1_unix'].apply(datetime.datetime.fromtimestamp)
df['ut1_unix_UTC']   = df['ut1_unix'].apply(datetime.datetime.utcfromtimestamp)

In [8]:
df

,year,month,day,hour,minute,second,recno,kindat,kinst,ut1_unix,...,tfreq,sn,smode,ssrc,pthlen,latcen,loncen,date,ut1_unix_LOCAL,ut1_unix_UTC
0,2018,12,18,00,00,00,0,17578,8308,1545112800,...,136000.0,NaN,'OPERA',PSK,0.0,52.8854,-2.6375,2018-12-18 00:00:00,2018-12-18 01:00:00,2018-12-18 06:00:00
1,2018,12,18,00,00,00,1,17578,8308,1545112800,...,3594000.0,-23.0,wspr,WSP,492.0,52.6411,3.5764,2018-12-18 00:00:00,2018-12-18 01:00:00,2018-12-18 06:00:00
2,2018,12,18,00,00,00,2,17578,8308,1545112800,...,3594002.0,-18.0,wspr,WSP,425.0,51.5944,6.5875,2018-12-18 00:00:00,2018-12-18 01:00:00,2018-12-18 06:00:00
3,2018,12,18,00,00,00,3,17578,8308,1545112800,...,3594002.0,-11.0,wspr,WSP,425.0,51.5944,6.5875,2018-12-18 00:00:00,2018-12-18 01:00:00,2018-12-18 06:00:00
4,2018,12,18,00,00,00,4,17578,8308,1545112800,...,3594002.0,-23.0,wspr,WSP,553.0,53.1435,2.8551,2018-12-18 00:00:00,2018-12-18 01:00:00,2018-12-18 06:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10470870,2018,12,18,23,59,59,10470870,17578,8308,1545199199,...,18101831.0,-15.0,'FT8',PSK,9964.2,4.5173,-89.4526,2018-12-18 23:59:59,2018-12-19 00:59:59,2018-12-19 05:59:59
10470871,2018,12,18,23,59,59,10470871,17578,8308,1545199199,...,18101830.0,-10.0,'FT8',PSK,9975.1,4.3132,-89.8139,2018-12-18 23:59:59,2018-12-19 00:59:59,2018-12-19 05:59:59
10470872,2018,12,18,23,59,59,10470872,17578,8308,1545199199,...,14075831.0,10.0,'FT8',PSK,7541.6,2.1346,-69.1488,2018-12-18 23:59:59,2018-12-19 00:59:59,2018-12-19 05:59:59
10470873,2018,12,18,23,59,59,10470873,17578,8308,1545199199,...,10136778.0,-9.0,'FT8',PSK,714.8,34.6422,-82.6012,2018-12-18 23:59:59,2018-12-19 00:59:59,2018-12-19 05:59:59


## Compare `ut1_unix` and `ut2_unix`

In [10]:
df[['date','ut1_unix','ut2_unix']]

,date,ut1_unix,ut2_unix
0,2018-12-18 00:00:00,1545112800,1545112800
1,2018-12-18 00:00:00,1545112800,1545112800
2,2018-12-18 00:00:00,1545112800,1545112800
3,2018-12-18 00:00:00,1545112800,1545112800
4,2018-12-18 00:00:00,1545112800,1545112800
...,...,...,...
10470870,2018-12-18 23:59:59,1545199199,1545199199
10470871,2018-12-18 23:59:59,1545199199,1545199199
10470872,2018-12-18 23:59:59,1545199199,1545199199
10470873,2018-12-18 23:59:59,1545199199,1545199199


## Compare Coversion of `ut1_unix` with known UTC Datetime

In [12]:
df[['date','ut1_unix_UTC','ut1_unix_LOCAL']]

,date,ut1_unix_UTC,ut1_unix_LOCAL
0,2018-12-18 00:00:00,2018-12-18 06:00:00,2018-12-18 01:00:00
1,2018-12-18 00:00:00,2018-12-18 06:00:00,2018-12-18 01:00:00
2,2018-12-18 00:00:00,2018-12-18 06:00:00,2018-12-18 01:00:00
3,2018-12-18 00:00:00,2018-12-18 06:00:00,2018-12-18 01:00:00
4,2018-12-18 00:00:00,2018-12-18 06:00:00,2018-12-18 01:00:00
...,...,...,...
10470870,2018-12-18 23:59:59,2018-12-19 05:59:59,2018-12-19 00:59:59
10470871,2018-12-18 23:59:59,2018-12-19 05:59:59,2018-12-19 00:59:59
10470872,2018-12-18 23:59:59,2018-12-19 05:59:59,2018-12-19 00:59:59
10470873,2018-12-18 23:59:59,2018-12-19 05:59:59,2018-12-19 00:59:59
